In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.home()
    / "projects"
    / "chart_data_retrieval"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Project exists:", PROJECT_ROOT.exists())

Project root: /home/pnola130/projects/chart_data_retrieval
Project exists: True


In [1]:
# ============================================================
# BLOCK 19 — Version 2 task-specific fusion architecture
# Baseline-preserving initialization
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = (
    Path.home()
    / "projects"
    / "chart_data_retrieval"
)

FUSION_INPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "embeddings"
    / "fusion_inputs"
)

FUSION_TRIPLET_PATH = (
    PROJECT_ROOT
    / "data"
    / "derived"
    / "fusion"
    / "fusion_triplets.parquet"
)

V2_MODEL_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "models"
    / "task_specific_fusion_v2"
)

V2_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "experiments"
    / "v2_task_specific_heads"
)

V2_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

V2_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# Load fixed semantic and graph embeddings
# ------------------------------------------------------------

chart_semantic_embeddings = np.load(
    FUSION_INPUT_DIR
    / "chart_semantic_embeddings.npy"
).astype("float32")

chart_graph_embeddings = np.load(
    FUSION_INPUT_DIR
    / "chart_graph_embeddings.npy"
).astype("float32")

table_semantic_embeddings = np.load(
    FUSION_INPUT_DIR
    / "table_semantic_embeddings.npy"
).astype("float32")

table_graph_embeddings = np.load(
    FUSION_INPUT_DIR
    / "table_graph_embeddings.npy"
).astype("float32")

query_semantic_embeddings = np.load(
    FUSION_INPUT_DIR
    / "query_semantic_embeddings.npy"
).astype("float32")


chart_metadata_df = pd.read_parquet(
    FUSION_INPUT_DIR
    / "chart_metadata.parquet"
).reset_index(drop=True)

table_metadata_df = pd.read_parquet(
    FUSION_INPUT_DIR
    / "table_metadata.parquet"
).reset_index(drop=True)

query_metadata_df = pd.read_parquet(
    FUSION_INPUT_DIR
    / "query_metadata.parquet"
).reset_index(drop=True)

fusion_triplets_df = pd.read_parquet(
    FUSION_TRIPLET_PATH
)


# ------------------------------------------------------------
# Validate source embeddings
# ------------------------------------------------------------

assert chart_semantic_embeddings.shape == (
    4104,
    768
)

assert chart_graph_embeddings.shape == (
    4104,
    256
)

assert table_semantic_embeddings.shape == (
    1400,
    768
)

assert table_graph_embeddings.shape == (
    1400,
    256
)

assert query_semantic_embeddings.shape == (
    986,
    768
)


print("\nInput embeddings")

print(
    "Chart semantic:",
    chart_semantic_embeddings.shape
)

print(
    "Chart graph:",
    chart_graph_embeddings.shape
)

print(
    "Table semantic:",
    table_semantic_embeddings.shape
)

print(
    "Table graph:",
    table_graph_embeddings.shape
)

print(
    "Query semantic:",
    query_semantic_embeddings.shape
)


print("\nTraining triplets")

print(
    fusion_triplets_df.groupby(
        [
            "task",
            "split"
        ]
    ).size()
)


# ------------------------------------------------------------
# Convert fixed features to tensors
# Keep them on CPU; batches will be moved to GPU.
# ------------------------------------------------------------

chart_semantic_tensor = torch.from_numpy(
    chart_semantic_embeddings
)

chart_graph_tensor = torch.from_numpy(
    chart_graph_embeddings
)

table_semantic_tensor = torch.from_numpy(
    table_semantic_embeddings
)

table_graph_tensor = torch.from_numpy(
    table_graph_embeddings
)

query_semantic_tensor = torch.from_numpy(
    query_semantic_embeddings
)


# ------------------------------------------------------------
# Zero-initialized residual adapter
#
# The last layer begins at zero, so the initial model
# produces exactly the original baseline embedding.
# ------------------------------------------------------------

class ZeroInitializedAdapter(nn.Module):

    def __init__(
        self,
        input_dimension,
        output_dimension,
        hidden_dimension=512,
        dropout=0.10
    ):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dimension,
                hidden_dimension
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                hidden_dimension,
                output_dimension
            )
        )

        final_layer = self.network[-1]

        nn.init.zeros_(
            final_layer.weight
        )

        nn.init.zeros_(
            final_layer.bias
        )

    def forward(self, features):

        return self.network(features)


# ------------------------------------------------------------
# Task-specific fusion model
# ------------------------------------------------------------

class TaskSpecificFusionModel(nn.Module):

    def __init__(
        self,
        semantic_dimension=768,
        graph_dimension=256,
        hidden_dimension=512,
        dropout=0.10
    ):

        super().__init__()

        # ====================================================
        # Task 1: Query → Chart
        #
        # Query remains in the original E5 space.
        # Only the chart receives a graph-informed residual.
        # Therefore initial performance equals DePlot+E5.
        # ====================================================

        self.query_chart_graph_adapter = (
            ZeroInitializedAdapter(
                input_dimension=graph_dimension,
                output_dimension=semantic_dimension,
                hidden_dimension=hidden_dimension,
                dropout=dropout
            )
        )

        self.query_chart_semantic_adapter = (
            ZeroInitializedAdapter(
                input_dimension=semantic_dimension,
                output_dimension=semantic_dimension,
                hidden_dimension=hidden_dimension,
                dropout=dropout
            )
        )

        self.query_chart_graph_scale = (
            nn.Parameter(
                torch.tensor(-2.9444)
            )
        )

        self.query_chart_semantic_scale = (
            nn.Parameter(
                torch.tensor(-2.9444)
            )
        )

        # ====================================================
        # Task 2: Chart → Chart
        #
        # Start from the existing ChartKG graph embedding.
        # Semantic information is added only as a residual.
        # ====================================================

        self.chart_chart_semantic_adapter = (
            ZeroInitializedAdapter(
                input_dimension=semantic_dimension,
                output_dimension=graph_dimension,
                hidden_dimension=hidden_dimension,
                dropout=dropout
            )
        )

        self.chart_chart_graph_adapter = (
            ZeroInitializedAdapter(
                input_dimension=graph_dimension,
                output_dimension=graph_dimension,
                hidden_dimension=hidden_dimension,
                dropout=dropout
            )
        )

        self.chart_chart_semantic_scale = (
            nn.Parameter(
                torch.tensor(-2.9444)
            )
        )

        self.chart_chart_graph_scale = (
            nn.Parameter(
                torch.tensor(-2.9444)
            )
        )

        # ====================================================
        # Task 3: Chart → Table
        #
        # Separate chart and table adapters.
        # Both begin from their ChartKG graph embeddings.
        # ====================================================

        self.chart_table_chart_semantic_adapter = (
            ZeroInitializedAdapter(
                input_dimension=semantic_dimension,
                output_dimension=graph_dimension,
                hidden_dimension=hidden_dimension,
                dropout=dropout
            )
        )

        self.chart_table_chart_graph_adapter = (
            ZeroInitializedAdapter(
                input_dimension=graph_dimension,
                output_dimension=graph_dimension,
                hidden_dimension=hidden_dimension,
                dropout=dropout
            )
        )

        self.chart_table_table_semantic_adapter = (
            ZeroInitializedAdapter(
                input_dimension=semantic_dimension,
                output_dimension=graph_dimension,
                hidden_dimension=hidden_dimension,
                dropout=dropout
            )
        )

        self.chart_table_table_graph_adapter = (
            ZeroInitializedAdapter(
                input_dimension=graph_dimension,
                output_dimension=graph_dimension,
                hidden_dimension=hidden_dimension,
                dropout=dropout
            )
        )

        self.chart_table_semantic_scale = (
            nn.Parameter(
                torch.tensor(-2.9444)
            )
        )

        self.chart_table_graph_scale = (
            nn.Parameter(
                torch.tensor(-2.9444)
            )
        )

    # --------------------------------------------------------
    # Query → Chart encoders
    # --------------------------------------------------------

    def encode_query_for_chart(
        self,
        query_semantic
    ):

        # Keep E5 query geometry fixed.
        return F.normalize(
            query_semantic,
            p=2,
            dim=-1
        )

    def encode_chart_for_query(
        self,
        chart_semantic,
        chart_graph
    ):

        semantic_scale = torch.sigmoid(
            self.query_chart_semantic_scale
        )

        graph_scale = torch.sigmoid(
            self.query_chart_graph_scale
        )

        representation = (
            chart_semantic
            + semantic_scale
            * self.query_chart_semantic_adapter(
                chart_semantic
            )
            + graph_scale
            * self.query_chart_graph_adapter(
                chart_graph
            )
        )

        return F.normalize(
            representation,
            p=2,
            dim=-1
        )

    # --------------------------------------------------------
    # Chart → Chart encoder
    # --------------------------------------------------------

    def encode_chart_for_chart(
        self,
        chart_semantic,
        chart_graph
    ):

        semantic_scale = torch.sigmoid(
            self.chart_chart_semantic_scale
        )

        graph_scale = torch.sigmoid(
            self.chart_chart_graph_scale
        )

        representation = (
            chart_graph
            + semantic_scale
            * self.chart_chart_semantic_adapter(
                chart_semantic
            )
            + graph_scale
            * self.chart_chart_graph_adapter(
                chart_graph
            )
        )

        return F.normalize(
            representation,
            p=2,
            dim=-1
        )

    # --------------------------------------------------------
    # Chart → Table encoders
    # --------------------------------------------------------

    def encode_chart_for_table(
        self,
        chart_semantic,
        chart_graph
    ):

        semantic_scale = torch.sigmoid(
            self.chart_table_semantic_scale
        )

        graph_scale = torch.sigmoid(
            self.chart_table_graph_scale
        )

        representation = (
            chart_graph
            + semantic_scale
            * self.chart_table_chart_semantic_adapter(
                chart_semantic
            )
            + graph_scale
            * self.chart_table_chart_graph_adapter(
                chart_graph
            )
        )

        return F.normalize(
            representation,
            p=2,
            dim=-1
        )

    def encode_table_for_chart(
        self,
        table_semantic,
        table_graph
    ):

        semantic_scale = torch.sigmoid(
            self.chart_table_semantic_scale
        )

        graph_scale = torch.sigmoid(
            self.chart_table_graph_scale
        )

        representation = (
            table_graph
            + semantic_scale
            * self.chart_table_table_semantic_adapter(
                table_semantic
            )
            + graph_scale
            * self.chart_table_table_graph_adapter(
                table_graph
            )
        )

        return F.normalize(
            representation,
            p=2,
            dim=-1
        )


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

v2_model = TaskSpecificFusionModel(
    semantic_dimension=768,
    graph_dimension=256,
    hidden_dimension=512,
    dropout=0.10
).to(device)


trainable_parameters = sum(
    parameter.numel()
    for parameter in v2_model.parameters()
    if parameter.requires_grad
)


print(
    "\nTrainable parameters:",
    f"{trainable_parameters:,}"
)


# ------------------------------------------------------------
# Smoke test: initial outputs must match baselines
# ------------------------------------------------------------

SMOKE_BATCH_SIZE = 64

with torch.inference_mode():

    chart_semantic_batch = (
        chart_semantic_tensor[
            :SMOKE_BATCH_SIZE
        ].to(device)
    )

    chart_graph_batch = (
        chart_graph_tensor[
            :SMOKE_BATCH_SIZE
        ].to(device)
    )

    table_semantic_batch = (
        table_semantic_tensor[
            :SMOKE_BATCH_SIZE
        ].to(device)
    )

    table_graph_batch = (
        table_graph_tensor[
            :SMOKE_BATCH_SIZE
        ].to(device)
    )

    query_semantic_batch = (
        query_semantic_tensor[
            :SMOKE_BATCH_SIZE
        ].to(device)
    )

    query_output = (
        v2_model.encode_query_for_chart(
            query_semantic_batch
        )
    )

    query_chart_output = (
        v2_model.encode_chart_for_query(
            chart_semantic_batch,
            chart_graph_batch
        )
    )

    chart_chart_output = (
        v2_model.encode_chart_for_chart(
            chart_semantic_batch,
            chart_graph_batch
        )
    )

    chart_table_chart_output = (
        v2_model.encode_chart_for_table(
            chart_semantic_batch,
            chart_graph_batch
        )
    )

    chart_table_table_output = (
        v2_model.encode_table_for_chart(
            table_semantic_batch,
            table_graph_batch
        )
    )


query_baseline = F.normalize(
    query_semantic_batch,
    p=2,
    dim=-1
)

chart_semantic_baseline = F.normalize(
    chart_semantic_batch,
    p=2,
    dim=-1
)

chart_graph_baseline = F.normalize(
    chart_graph_batch,
    p=2,
    dim=-1
)

table_graph_baseline = F.normalize(
    table_graph_batch,
    p=2,
    dim=-1
)


initialization_checks = {
    "query_matches_E5": float(
        F.cosine_similarity(
            query_output,
            query_baseline
        ).mean()
    ),
    "query_chart_matches_DePlot_E5": float(
        F.cosine_similarity(
            query_chart_output,
            chart_semantic_baseline
        ).mean()
    ),
    "chart_chart_matches_ChartKG": float(
        F.cosine_similarity(
            chart_chart_output,
            chart_graph_baseline
        ).mean()
    ),
    "chart_table_chart_matches_ChartKG": float(
        F.cosine_similarity(
            chart_table_chart_output,
            chart_graph_baseline
        ).mean()
    ),
    "chart_table_table_matches_ChartKG": float(
        F.cosine_similarity(
            chart_table_table_output,
            table_graph_baseline
        ).mean()
    )
}


print("\nBaseline-preserving initialization")

for check_name, value in (
    initialization_checks.items()
):

    print(
        check_name,
        ":",
        round(value, 8)
    )


for value in initialization_checks.values():

    assert value > 0.99999, (
        "Initialization changed a baseline "
        "embedding unexpectedly."
    )


# ------------------------------------------------------------
# Save initial architecture
# ------------------------------------------------------------

INITIAL_MODEL_PATH = (
    V2_MODEL_DIR
    / "initial_model.pt"
)

torch.save(
    {
        "model_state_dict": (
            v2_model.state_dict()
        ),
        "architecture": (
            "TaskSpecificFusionModel"
        ),
        "initialization_checks": (
            initialization_checks
        ),
        "trainable_parameters": (
            trainable_parameters
        )
    },
    INITIAL_MODEL_PATH
)


v2_configuration = {
    "experiment": (
        "v2_task_specific_heads"
    ),
    "query_chart_initialization": (
        "DePlot+E5"
    ),
    "chart_chart_initialization": (
        "ChartKG R-GCN"
    ),
    "chart_table_initialization": (
        "ChartKG R-GCN"
    ),
    "query_embedding_dimension": 768,
    "chart_chart_dimension": 256,
    "chart_table_dimension": 256,
    "shared_backbone": False,
    "shared_fixed_input_encoders": True,
    "task_specific_heads": True,
    "query_encoder_frozen": True,
    "planned_checkpoint_selection": (
        "full validation retrieval metrics"
    )
}

with open(
    V2_MODEL_DIR
    / "architecture_config.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        v2_configuration,
        file,
        indent=2
    )


print("\nVersion 2 architecture ready")

print(
    "Initial model:",
    INITIAL_MODEL_PATH
)

print(
    "Configuration:",
    V2_MODEL_DIR
    / "architecture_config.json"
)

Device: cuda
GPU: NVIDIA A100-SXM4-80GB

Input embeddings
Chart semantic: (4104, 768)
Chart graph: (4104, 256)
Table semantic: (1400, 768)
Table graph: (1400, 256)
Query semantic: (986, 768)

Training triplets
task         split
chart_chart  test      394
             train    1918
             val       392
chart_table  test      347
             train    1709
             val       346
query_chart  test      518
             train    2370
             val       525
dtype: int64

Trainable parameters: 3,677,190

Baseline-preserving initialization
query_matches_E5 : 1.0
query_chart_matches_DePlot_E5 : 1.0
chart_chart_matches_ChartKG : 1.0
chart_table_chart_matches_ChartKG : 1.0
chart_table_table_matches_ChartKG : 1.0

Version 2 architecture ready
Initial model: /home/pnola130/projects/chart_data_retrieval/artifacts/models/task_specific_fusion_v2/initial_model.pt
Configuration: /home/pnola130/projects/chart_data_retrieval/artifacts/models/task_specific_fusion_v2/architecture_config.json

In [2]:
# ============================================================
# BLOCK 20 — V2 loaders and baseline validation
# Do not train yet
# ============================================================

from torch.utils.data import (
    Dataset,
    DataLoader
)

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Data identities
#
# A shuffled chart/table has the same source_data_id,
# but it represents different underlying data.
# Therefore it must receive a different identity label.
# ------------------------------------------------------------

chart_identity_strings = np.where(
    chart_metadata_df[
        "variant_type"
    ].astype(str).to_numpy()
    == "hard_negative",
    (
        chart_metadata_df[
            "source_data_id"
        ].astype(str).to_numpy()
        + "__counterfactual"
    ),
    chart_metadata_df[
        "source_data_id"
    ].astype(str).to_numpy()
)

table_identity_strings = np.where(
    table_metadata_df[
        "table_variant_type"
    ].astype(str).to_numpy()
    == "hard_negative",
    (
        table_metadata_df[
            "source_data_id"
        ].astype(str).to_numpy()
        + "__counterfactual"
    ),
    table_metadata_df[
        "source_data_id"
    ].astype(str).to_numpy()
)

query_identity_strings = (
    query_metadata_df[
        "data_id"
    ].astype(str).to_numpy()
)


# Create one shared integer identity vocabulary
all_identity_strings = np.concatenate(
    [
        query_identity_strings,
        chart_identity_strings,
        table_identity_strings
    ]
)

unique_identity_strings = sorted(
    set(all_identity_strings)
)

identity_to_integer = {
    identity: index
    for index, identity in enumerate(
        unique_identity_strings
    )
}

query_identity_labels = np.asarray(
    [
        identity_to_integer[value]
        for value in query_identity_strings
    ],
    dtype=np.int64
)

chart_identity_labels = np.asarray(
    [
        identity_to_integer[value]
        for value in chart_identity_strings
    ],
    dtype=np.int64
)

table_identity_labels = np.asarray(
    [
        identity_to_integer[value]
        for value in table_identity_strings
    ],
    dtype=np.int64
)


print(
    "Unique underlying data identities:",
    len(unique_identity_strings)
)


# ------------------------------------------------------------
# Triplet-index dataset
# ------------------------------------------------------------

class TaskTripletDataset(Dataset):

    def __init__(
        self,
        triplet_dataframe
    ):

        self.records = (
            triplet_dataframe[
                [
                    "anchor_index",
                    "positive_index",
                    "negative_index"
                ]
            ]
            .astype("int64")
            .to_numpy()
        )

    def __len__(self):

        return len(self.records)

    def __getitem__(self, index):

        anchor_index, positive_index, negative_index = (
            self.records[index]
        )

        return {
            "anchor_index": torch.tensor(
                anchor_index,
                dtype=torch.long
            ),
            "positive_index": torch.tensor(
                positive_index,
                dtype=torch.long
            ),
            "negative_index": torch.tensor(
                negative_index,
                dtype=torch.long
            )
        }


# ------------------------------------------------------------
# Create separate task datasets
# ------------------------------------------------------------

TASKS = [
    "query_chart",
    "chart_chart",
    "chart_table"
]

train_task_frames = {}
validation_task_frames = {}

train_datasets = {}
validation_datasets = {}

train_loaders = {}
validation_loaders = {}


TRAIN_BATCH_SIZE = 256
VALIDATION_BATCH_SIZE = 512


for task_name in TASKS:

    train_task_frames[task_name] = (
        fusion_triplets_df[
            (
                fusion_triplets_df["task"]
                == task_name
            )
            &
            (
                fusion_triplets_df["split"]
                == "train"
            )
        ]
        .reset_index(drop=True)
    )

    validation_task_frames[task_name] = (
        fusion_triplets_df[
            (
                fusion_triplets_df["task"]
                == task_name
            )
            &
            (
                fusion_triplets_df["split"]
                == "val"
            )
        ]
        .reset_index(drop=True)
    )

    train_datasets[task_name] = (
        TaskTripletDataset(
            train_task_frames[
                task_name
            ]
        )
    )

    validation_datasets[task_name] = (
        TaskTripletDataset(
            validation_task_frames[
                task_name
            ]
        )
    )

    train_loaders[task_name] = DataLoader(
        train_datasets[task_name],
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(
            device.type == "cuda"
        ),
        drop_last=False
    )

    validation_loaders[task_name] = DataLoader(
        validation_datasets[
            task_name
        ],
        batch_size=VALIDATION_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(
            device.type == "cuda"
        ),
        drop_last=False
    )


print("\nTask loaders")

for task_name in TASKS:

    print(
        task_name,
        "| train examples:",
        len(train_datasets[task_name]),
        "| train batches:",
        len(train_loaders[task_name]),
        "| validation examples:",
        len(
            validation_datasets[
                task_name
            ]
        )
    )


# ------------------------------------------------------------
# Confirm parameter separation
# ------------------------------------------------------------

query_chart_parameters = [
    parameter
    for name, parameter
    in v2_model.named_parameters()
    if name.startswith(
        "query_chart_"
    )
]

chart_chart_parameters = [
    parameter
    for name, parameter
    in v2_model.named_parameters()
    if name.startswith(
        "chart_chart_"
    )
]

chart_table_parameters = [
    parameter
    for name, parameter
    in v2_model.named_parameters()
    if name.startswith(
        "chart_table_"
    )
]


def count_parameters(parameters):

    return sum(
        parameter.numel()
        for parameter in parameters
        if parameter.requires_grad
    )


print("\nTask-specific parameters")

print(
    "Query → Chart:",
    f"{count_parameters(query_chart_parameters):,}"
)

print(
    "Chart → Chart:",
    f"{count_parameters(chart_chart_parameters):,}"
)

print(
    "Chart → Table:",
    f"{count_parameters(chart_table_parameters):,}"
)


# ------------------------------------------------------------
# Batched encoding helper
# ------------------------------------------------------------

def encode_in_batches(
    number_of_items,
    encoding_function,
    batch_size=512,
    description="Encoding"
):

    encoded_batches = []

    with torch.inference_mode():

        for start in tqdm(
            range(
                0,
                number_of_items,
                batch_size
            ),
            desc=description
        ):

            end = min(
                start + batch_size,
                number_of_items
            )

            encoded_batch = (
                encoding_function(
                    start,
                    end
                )
            )

            encoded_batches.append(
                encoded_batch
                .detach()
                .cpu()
                .numpy()
                .astype("float32")
            )

    return np.concatenate(
        encoded_batches,
        axis=0
    )


# ------------------------------------------------------------
# Full embedding generation for one task
# ------------------------------------------------------------

def generate_task_embeddings(
    model,
    task_name
):

    model.eval()

    if task_name == "query_chart":

        query_outputs = encode_in_batches(
            number_of_items=len(
                query_semantic_tensor
            ),
            encoding_function=lambda start, end: (
                model.encode_query_for_chart(
                    query_semantic_tensor[
                        start:end
                    ].to(device)
                )
            ),
            description=(
                "Encoding query→chart queries"
            )
        )

        chart_outputs = encode_in_batches(
            number_of_items=len(
                chart_semantic_tensor
            ),
            encoding_function=lambda start, end: (
                model.encode_chart_for_query(
                    chart_semantic_tensor[
                        start:end
                    ].to(device),
                    chart_graph_tensor[
                        start:end
                    ].to(device)
                )
            ),
            description=(
                "Encoding query→chart charts"
            )
        )

        return {
            "query": query_outputs,
            "chart": chart_outputs
        }

    if task_name == "chart_chart":

        chart_outputs = encode_in_batches(
            number_of_items=len(
                chart_semantic_tensor
            ),
            encoding_function=lambda start, end: (
                model.encode_chart_for_chart(
                    chart_semantic_tensor[
                        start:end
                    ].to(device),
                    chart_graph_tensor[
                        start:end
                    ].to(device)
                )
            ),
            description=(
                "Encoding chart→chart charts"
            )
        )

        return {
            "chart": chart_outputs
        }

    if task_name == "chart_table":

        chart_outputs = encode_in_batches(
            number_of_items=len(
                chart_semantic_tensor
            ),
            encoding_function=lambda start, end: (
                model.encode_chart_for_table(
                    chart_semantic_tensor[
                        start:end
                    ].to(device),
                    chart_graph_tensor[
                        start:end
                    ].to(device)
                )
            ),
            description=(
                "Encoding chart→table charts"
            )
        )

        table_outputs = encode_in_batches(
            number_of_items=len(
                table_semantic_tensor
            ),
            encoding_function=lambda start, end: (
                model.encode_table_for_chart(
                    table_semantic_tensor[
                        start:end
                    ].to(device),
                    table_graph_tensor[
                        start:end
                    ].to(device)
                )
            ),
            description=(
                "Encoding chart→table tables"
            )
        )

        return {
            "chart": chart_outputs,
            "table": table_outputs
        }

    raise ValueError(
        f"Unknown task: {task_name}"
    )


# ------------------------------------------------------------
# Ranking metric helper
# ------------------------------------------------------------

def evaluate_rankings(
    similarity_matrix,
    positive_masks,
    counterfactual_masks,
    self_masks=None
):

    result_records = []

    for query_position in range(
        similarity_matrix.shape[0]
    ):

        similarities = (
            similarity_matrix[
                query_position
            ].copy()
        )

        positive_mask = (
            positive_masks[
                query_position
            ]
        )

        counterfactual_mask = (
            counterfactual_masks[
                query_position
            ]
        )

        if self_masks is not None:

            self_mask = (
                self_masks[
                    query_position
                ]
            )

            similarities[
                self_mask
            ] = -np.inf

            positive_mask = (
                positive_mask
                & (~self_mask)
            )

        number_of_positives = int(
            positive_mask.sum()
        )

        if number_of_positives == 0:
            continue

        ranking = np.argsort(
            -similarities,
            kind="stable"
        )

        ranked_relevance = (
            positive_mask[ranking]
        )

        positive_ranks = (
            np.flatnonzero(
                ranked_relevance
            )
            + 1
        )

        first_positive_rank = int(
            positive_ranks[0]
        )

        best_positive_similarity = float(
            similarities[
                positive_mask
            ].max()
        )

        if counterfactual_mask.any():

            best_counterfactual_similarity = float(
                similarities[
                    counterfactual_mask
                ].max()
            )

            data_gap = (
                best_positive_similarity
                - best_counterfactual_similarity
            )

            data_accuracy = float(
                best_positive_similarity
                >
                best_counterfactual_similarity
            )

        else:

            best_counterfactual_similarity = np.nan
            data_gap = np.nan
            data_accuracy = np.nan

        relevance_at_10 = (
            ranked_relevance[:10]
            .astype(np.float32)
        )

        discounts = np.log2(
            np.arange(
                2,
                len(relevance_at_10) + 2
            )
        )

        dcg = float(
            np.sum(
                relevance_at_10
                / discounts
            )
        )

        ideal_length = min(
            number_of_positives,
            10
        )

        ideal_discounts = np.log2(
            np.arange(
                2,
                ideal_length + 2
            )
        )

        idcg = float(
            np.sum(
                np.ones(ideal_length)
                / ideal_discounts
            )
        )

        ndcg_at_10 = (
            dcg / idcg
            if idcg > 0
            else 0.0
        )

        result_records.append(
            {
                "first_positive_rank": (
                    first_positive_rank
                ),
                "recall_at_1": float(
                    first_positive_rank <= 1
                ),
                "recall_at_5": float(
                    first_positive_rank <= 5
                ),
                "recall_at_10": float(
                    first_positive_rank <= 10
                ),
                "reciprocal_rank": float(
                    1.0
                    / first_positive_rank
                ),
                "ndcg_at_10": (
                    ndcg_at_10
                ),
                "best_positive_similarity": (
                    best_positive_similarity
                ),
                "best_counterfactual_similarity": (
                    best_counterfactual_similarity
                ),
                "data_gap": data_gap,
                "data_accuracy": data_accuracy
            }
        )

    result_df = pd.DataFrame(
        result_records
    )

    return {
        "number_of_queries": int(
            len(result_df)
        ),
        "recall_at_1": float(
            result_df[
                "recall_at_1"
            ].mean()
        ),
        "recall_at_5": float(
            result_df[
                "recall_at_5"
            ].mean()
        ),
        "recall_at_10": float(
            result_df[
                "recall_at_10"
            ].mean()
        ),
        "mrr": float(
            result_df[
                "reciprocal_rank"
            ].mean()
        ),
        "ndcg_at_10": float(
            result_df[
                "ndcg_at_10"
            ].mean()
        ),
        "median_positive_rank": float(
            result_df[
                "first_positive_rank"
            ].median()
        ),
        "mean_data_gap": float(
            result_df[
                "data_gap"
            ].mean()
        ),
        "data_accuracy": float(
            result_df[
                "data_accuracy"
            ].mean()
        )
    }


# ------------------------------------------------------------
# Full validation retrieval for one task
# ------------------------------------------------------------

def evaluate_v2_validation(
    model,
    task_name
):

    task_embeddings = (
        generate_task_embeddings(
            model,
            task_name
        )
    )

    if task_name == "query_chart":

        query_indices = np.flatnonzero(
            query_metadata_df["split"]
            .astype(str)
            .to_numpy()
            == "val"
        )

        candidate_indices = np.flatnonzero(
            chart_metadata_df["split"]
            .astype(str)
            .to_numpy()
            == "val"
        )

        query_vectors = (
            task_embeddings["query"][
                query_indices
            ]
        )

        candidate_vectors = (
            task_embeddings["chart"][
                candidate_indices
            ]
        )

        query_sources = (
            query_metadata_df.iloc[
                query_indices
            ]["data_id"]
            .astype(str)
            .to_numpy()
        )

        candidate_sources = (
            chart_metadata_df.iloc[
                candidate_indices
            ]["source_data_id"]
            .astype(str)
            .to_numpy()
        )

        candidate_variants = (
            chart_metadata_df.iloc[
                candidate_indices
            ]["variant_type"]
            .astype(str)
            .to_numpy()
        )

        positive_masks = (
            (
                query_sources[:, None]
                == candidate_sources[None, :]
            )
            &
            (
                candidate_variants[None, :]
                != "hard_negative"
            )
        )

        counterfactual_masks = (
            (
                query_sources[:, None]
                == candidate_sources[None, :]
            )
            &
            (
                candidate_variants[None, :]
                == "hard_negative"
            )
        )

        similarity_matrix = (
            query_vectors
            @ candidate_vectors.T
        )

        return evaluate_rankings(
            similarity_matrix,
            positive_masks,
            counterfactual_masks
        )

    if task_name == "chart_chart":

        all_validation_indices = np.flatnonzero(
            chart_metadata_df["split"]
            .astype(str)
            .to_numpy()
            == "val"
        )

        query_indices = np.asarray(
            [
                index
                for index
                in all_validation_indices
                if (
                    str(
                        chart_metadata_df.iloc[
                            index
                        ]["variant_type"]
                    )
                    != "hard_negative"
                )
            ],
            dtype=np.int64
        )

        candidate_indices = (
            all_validation_indices
        )

        query_vectors = (
            task_embeddings["chart"][
                query_indices
            ]
        )

        candidate_vectors = (
            task_embeddings["chart"][
                candidate_indices
            ]
        )

        query_sources = (
            chart_metadata_df.iloc[
                query_indices
            ]["source_data_id"]
            .astype(str)
            .to_numpy()
        )

        candidate_sources = (
            chart_metadata_df.iloc[
                candidate_indices
            ]["source_data_id"]
            .astype(str)
            .to_numpy()
        )

        query_ids = (
            chart_metadata_df.iloc[
                query_indices
            ]["chart_id"]
            .astype(str)
            .to_numpy()
        )

        candidate_ids = (
            chart_metadata_df.iloc[
                candidate_indices
            ]["chart_id"]
            .astype(str)
            .to_numpy()
        )

        candidate_variants = (
            chart_metadata_df.iloc[
                candidate_indices
            ]["variant_type"]
            .astype(str)
            .to_numpy()
        )

        positive_masks = (
            (
                query_sources[:, None]
                == candidate_sources[None, :]
            )
            &
            (
                candidate_variants[None, :]
                != "hard_negative"
            )
        )

        counterfactual_masks = (
            (
                query_sources[:, None]
                == candidate_sources[None, :]
            )
            &
            (
                candidate_variants[None, :]
                == "hard_negative"
            )
        )

        self_masks = (
            query_ids[:, None]
            == candidate_ids[None, :]
        )

        similarity_matrix = (
            query_vectors
            @ candidate_vectors.T
        )

        return evaluate_rankings(
            similarity_matrix,
            positive_masks,
            counterfactual_masks,
            self_masks=self_masks
        )

    if task_name == "chart_table":

        query_indices = np.flatnonzero(
            (
                chart_metadata_df["split"]
                .astype(str)
                .to_numpy()
                == "val"
            )
            &
            (
                chart_metadata_df[
                    "variant_type"
                ].astype(str)
                .to_numpy()
                != "hard_negative"
            )
        )

        candidate_indices = np.flatnonzero(
            table_metadata_df["split"]
            .astype(str)
            .to_numpy()
            == "val"
        )

        query_vectors = (
            task_embeddings["chart"][
                query_indices
            ]
        )

        candidate_vectors = (
            task_embeddings["table"][
                candidate_indices
            ]
        )

        query_sources = (
            chart_metadata_df.iloc[
                query_indices
            ]["source_data_id"]
            .astype(str)
            .to_numpy()
        )

        candidate_sources = (
            table_metadata_df.iloc[
                candidate_indices
            ]["source_data_id"]
            .astype(str)
            .to_numpy()
        )

        candidate_variants = (
            table_metadata_df.iloc[
                candidate_indices
            ]["table_variant_type"]
            .astype(str)
            .to_numpy()
        )

        positive_masks = (
            (
                query_sources[:, None]
                == candidate_sources[None, :]
            )
            &
            (
                candidate_variants[None, :]
                == "original"
            )
        )

        counterfactual_masks = (
            (
                query_sources[:, None]
                == candidate_sources[None, :]
            )
            &
            (
                candidate_variants[None, :]
                == "hard_negative"
            )
        )

        similarity_matrix = (
            query_vectors
            @ candidate_vectors.T
        )

        return evaluate_rankings(
            similarity_matrix,
            positive_masks,
            counterfactual_masks
        )

    raise ValueError(
        f"Unknown task: {task_name}"
    )


# ------------------------------------------------------------
# Evaluate untrained Version 2
# ------------------------------------------------------------

initial_validation_metrics = {}

print("\nInitial full validation retrieval")

for task_name in TASKS:

    metrics = evaluate_v2_validation(
        v2_model,
        task_name
    )

    initial_validation_metrics[
        task_name
    ] = metrics

    print(f"\n{task_name}")

    display(
        pd.DataFrame(
            [metrics]
        ).round(6)
    )


# ------------------------------------------------------------
# Save starting metrics
# ------------------------------------------------------------

with open(
    V2_REPORT_DIR
    / "initial_validation_metrics.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        initial_validation_metrics,
        file,
        indent=2
    )


print(
    "\nSaved:",
    V2_REPORT_DIR
    / "initial_validation_metrics.json"
)

Unique underlying data identities: 1400

Task loaders
query_chart | train examples: 2370 | train batches: 10 | validation examples: 525
chart_chart | train examples: 1918 | train batches: 8 | validation examples: 392
chart_table | train examples: 1709 | train batches: 7 | validation examples: 346

Task-specific parameters
Query → Chart: 1,313,282
Chart → Chart: 787,970
Chart → Table: 1,575,938

Initial full validation retrieval


Encoding query→chart queries:   0%|          | 0/2 [00:00<?, ?it/s]

Encoding query→chart charts:   0%|          | 0/9 [00:00<?, ?it/s]


query_chart


,number_of_queries,recall_at_1,recall_at_5,recall_at_10,mrr,ndcg_at_10,median_positive_rank,mean_data_gap,data_accuracy
0,151,0.443709,0.754967,0.81457,0.579898,0.571538,2.0,0.001442,0.615894


Encoding chart→chart charts:   0%|          | 0/9 [00:00<?, ?it/s]

... (3 outputs left out)

Encoding chart→table tables:   0%|          | 0/3 [00:00<?, ?it/s]


chart_table


,number_of_queries,recall_at_1,recall_at_5,recall_at_10,mrr,ndcg_at_10,median_positive_rank,mean_data_gap,data_accuracy
0,346,0.956647,0.982659,0.982659,0.969122,0.971975,1.0,0.383341,0.962428



Saved: /home/pnola130/projects/chart_data_retrieval/reports/experiments/v2_task_specific_heads/initial_validation_metrics.json


In [3]:
# ============================================================
# V2 BLOCK 4 — Train task-specific heads
# ============================================================

from copy import deepcopy
import time


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


# ------------------------------------------------------------
# Identity-label tensors
# ------------------------------------------------------------

query_identity_tensor = torch.from_numpy(
    query_identity_labels
).long()

chart_identity_tensor = torch.from_numpy(
    chart_identity_labels
).long()

table_identity_tensor = torch.from_numpy(
    table_identity_labels
).long()


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

MAXIMUM_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 8
SCHEDULER_PATIENCE = 3

TASK_LEARNING_RATES = {
    "query_chart": 2e-4,
    "chart_chart": 1e-4,
    "chart_table": 1e-4
}

TASK_TEMPERATURES = {
    "query_chart": 0.07,
    "chart_chart": 0.07,
    "chart_table": 0.07
}

TASK_MARGINS = {
    "query_chart": 0.05,
    "chart_chart": 0.10,
    "chart_table": 0.10
}

INFO_NCE_WEIGHT = 1.0
RANKING_WEIGHT = 0.5
DISTILLATION_WEIGHT = 0.10

WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0


# ------------------------------------------------------------
# Task parameter prefixes
# ------------------------------------------------------------

TASK_PREFIXES = {
    "query_chart": "query_chart_",
    "chart_chart": "chart_chart_",
    "chart_table": "chart_table_"
}


# ------------------------------------------------------------
# Multi-positive InfoNCE
# ------------------------------------------------------------

def multi_positive_info_nce(
    anchor_embeddings,
    candidate_embeddings,
    anchor_labels,
    candidate_labels,
    temperature
):

    logits = (
        anchor_embeddings
        @ candidate_embeddings.T
    ) / temperature

    positive_mask = (
        anchor_labels[:, None]
        == candidate_labels[None, :]
    )

    valid_anchor_mask = (
        positive_mask.any(dim=1)
    )

    if not valid_anchor_mask.any():

        return torch.tensor(
            0.0,
            device=anchor_embeddings.device,
            requires_grad=True
        )

    negative_infinity = torch.finfo(
        logits.dtype
    ).min

    positive_logits = logits.masked_fill(
        ~positive_mask,
        negative_infinity
    )

    log_positive_sum = torch.logsumexp(
        positive_logits,
        dim=1
    )

    log_all_sum = torch.logsumexp(
        logits,
        dim=1
    )

    loss = -(
        log_positive_sum
        - log_all_sum
    )

    return loss[
        valid_anchor_mask
    ].mean()


# ------------------------------------------------------------
# Distillation helper
# ------------------------------------------------------------

def cosine_distillation_loss(
    learned_embeddings,
    baseline_embeddings
):

    baseline_embeddings = F.normalize(
        baseline_embeddings,
        p=2,
        dim=-1
    )

    return (
        1.0
        - F.cosine_similarity(
            learned_embeddings,
            baseline_embeddings,
            dim=-1
        )
    ).mean()


# ------------------------------------------------------------
# Freeze all parameters except one task head
# ------------------------------------------------------------

def activate_task_parameters(
    model,
    task_name
):

    task_prefix = TASK_PREFIXES[
        task_name
    ]

    active_parameters = []

    for name, parameter in (
        model.named_parameters()
    ):

        is_active = name.startswith(
            task_prefix
        )

        parameter.requires_grad = (
            is_active
        )

        if is_active:
            active_parameters.append(
                parameter
            )

    return active_parameters


# ------------------------------------------------------------
# Save and restore one task head
# ------------------------------------------------------------

def get_task_state(
    model,
    task_name
):

    task_prefix = TASK_PREFIXES[
        task_name
    ]

    return {
        name: parameter
        .detach()
        .cpu()
        .clone()
        for name, parameter
        in model.state_dict().items()
        if name.startswith(task_prefix)
    }


def load_task_state(
    model,
    task_state
):

    model_state = model.state_dict()

    for name, tensor in (
        task_state.items()
    ):

        model_state[name] = tensor.to(
            model_state[name].device
        )

    model.load_state_dict(
        model_state
    )


# ------------------------------------------------------------
# Retrieve and encode one triplet batch
# ------------------------------------------------------------

def encode_training_batch(
    model,
    task_name,
    batch
):

    anchor_indices = batch[
        "anchor_index"
    ].to(
        device,
        non_blocking=True
    )

    positive_indices = batch[
        "positive_index"
    ].to(
        device,
        non_blocking=True
    )

    negative_indices = batch[
        "negative_index"
    ].to(
        device,
        non_blocking=True
    )

    if task_name == "query_chart":

        anchor_semantic = (
            query_semantic_tensor[
                anchor_indices.cpu()
            ].to(
                device,
                non_blocking=True
            )
        )

        positive_semantic = (
            chart_semantic_tensor[
                positive_indices.cpu()
            ].to(
                device,
                non_blocking=True
            )
        )

        positive_graph = (
            chart_graph_tensor[
                positive_indices.cpu()
            ].to(
                device,
                non_blocking=True
            )
        )

        negative_semantic = (
            chart_semantic_tensor[
                negative_indices.cpu()
            ].to(
                device,
                non_blocking=True
            )
        )

        negative_graph = (
            chart_graph_tensor[
                negative_indices.cpu()
            ].to(
                device,
                non_blocking=True
            )
        )

        anchor_embeddings = (
            model.encode_query_for_chart(
                anchor_semantic
            )
        )

        positive_embeddings = (
            model.encode_chart_for_query(
                positive_semantic,
                positive_graph
            )
        )

        negative_embeddings = (
            model.encode_chart_for_query(
                negative_semantic,
                negative_graph
            )
        )

        anchor_labels = (
            query_identity_tensor[
                anchor_indices.cpu()
            ].to(device)
        )

        positive_labels = (
            chart_identity_tensor[
                positive_indices.cpu()
            ].to(device)
        )

        negative_labels = (
            chart_identity_tensor[
                negative_indices.cpu()
            ].to(device)
        )

        distillation_loss = (
            0.5
            * cosine_distillation_loss(
                positive_embeddings,
                positive_semantic
            )
            +
            0.5
            * cosine_distillation_loss(
                negative_embeddings,
                negative_semantic
            )
        )

    elif task_name == "chart_chart":

        anchor_semantic = (
            chart_semantic_tensor[
                anchor_indices.cpu()
            ].to(device)
        )

        anchor_graph = (
            chart_graph_tensor[
                anchor_indices.cpu()
            ].to(device)
        )

        positive_semantic = (
            chart_semantic_tensor[
                positive_indices.cpu()
            ].to(device)
        )

        positive_graph = (
            chart_graph_tensor[
                positive_indices.cpu()
            ].to(device)
        )

        negative_semantic = (
            chart_semantic_tensor[
                negative_indices.cpu()
            ].to(device)
        )

        negative_graph = (
            chart_graph_tensor[
                negative_indices.cpu()
            ].to(device)
        )

        anchor_embeddings = (
            model.encode_chart_for_chart(
                anchor_semantic,
                anchor_graph
            )
        )

        positive_embeddings = (
            model.encode_chart_for_chart(
                positive_semantic,
                positive_graph
            )
        )

        negative_embeddings = (
            model.encode_chart_for_chart(
                negative_semantic,
                negative_graph
            )
        )

        anchor_labels = (
            chart_identity_tensor[
                anchor_indices.cpu()
            ].to(device)
        )

        positive_labels = (
            chart_identity_tensor[
                positive_indices.cpu()
            ].to(device)
        )

        negative_labels = (
            chart_identity_tensor[
                negative_indices.cpu()
            ].to(device)
        )

        distillation_loss = (
            (
                cosine_distillation_loss(
                    anchor_embeddings,
                    anchor_graph
                )
                +
                cosine_distillation_loss(
                    positive_embeddings,
                    positive_graph
                )
                +
                cosine_distillation_loss(
                    negative_embeddings,
                    negative_graph
                )
            )
            / 3.0
        )

    elif task_name == "chart_table":

        anchor_semantic = (
            chart_semantic_tensor[
                anchor_indices.cpu()
            ].to(device)
        )

        anchor_graph = (
            chart_graph_tensor[
                anchor_indices.cpu()
            ].to(device)
        )

        positive_semantic = (
            table_semantic_tensor[
                positive_indices.cpu()
            ].to(device)
        )

        positive_graph = (
            table_graph_tensor[
                positive_indices.cpu()
            ].to(device)
        )

        negative_semantic = (
            table_semantic_tensor[
                negative_indices.cpu()
            ].to(device)
        )

        negative_graph = (
            table_graph_tensor[
                negative_indices.cpu()
            ].to(device)
        )

        anchor_embeddings = (
            model.encode_chart_for_table(
                anchor_semantic,
                anchor_graph
            )
        )

        positive_embeddings = (
            model.encode_table_for_chart(
                positive_semantic,
                positive_graph
            )
        )

        negative_embeddings = (
            model.encode_table_for_chart(
                negative_semantic,
                negative_graph
            )
        )

        anchor_labels = (
            chart_identity_tensor[
                anchor_indices.cpu()
            ].to(device)
        )

        positive_labels = (
            table_identity_tensor[
                positive_indices.cpu()
            ].to(device)
        )

        negative_labels = (
            table_identity_tensor[
                negative_indices.cpu()
            ].to(device)
        )

        distillation_loss = (
            (
                cosine_distillation_loss(
                    anchor_embeddings,
                    anchor_graph
                )
                +
                cosine_distillation_loss(
                    positive_embeddings,
                    positive_graph
                )
                +
                cosine_distillation_loss(
                    negative_embeddings,
                    negative_graph
                )
            )
            / 3.0
        )

    else:

        raise ValueError(
            f"Unknown task: {task_name}"
        )

    return {
        "anchor_embeddings": (
            anchor_embeddings
        ),
        "positive_embeddings": (
            positive_embeddings
        ),
        "negative_embeddings": (
            negative_embeddings
        ),
        "anchor_labels": anchor_labels,
        "positive_labels": (
            positive_labels
        ),
        "negative_labels": (
            negative_labels
        ),
        "distillation_loss": (
            distillation_loss
        )
    }


# ------------------------------------------------------------
# Calculate training loss
# ------------------------------------------------------------

def calculate_task_loss(
    encoded_batch,
    task_name
):

    anchor_embeddings = encoded_batch[
        "anchor_embeddings"
    ]

    positive_embeddings = encoded_batch[
        "positive_embeddings"
    ]

    negative_embeddings = encoded_batch[
        "negative_embeddings"
    ]

    anchor_labels = encoded_batch[
        "anchor_labels"
    ]

    positive_labels = encoded_batch[
        "positive_labels"
    ]

    negative_labels = encoded_batch[
        "negative_labels"
    ]

    candidate_embeddings = torch.cat(
        [
            positive_embeddings,
            negative_embeddings
        ],
        dim=0
    )

    candidate_labels = torch.cat(
        [
            positive_labels,
            negative_labels
        ],
        dim=0
    )

    info_nce_loss = (
        multi_positive_info_nce(
            anchor_embeddings,
            candidate_embeddings,
            anchor_labels,
            candidate_labels,
            temperature=(
                TASK_TEMPERATURES[
                    task_name
                ]
            )
        )
    )

    positive_similarity = (
        anchor_embeddings
        * positive_embeddings
    ).sum(dim=-1)

    negative_similarity = (
        anchor_embeddings
        * negative_embeddings
    ).sum(dim=-1)

    ranking_loss = F.relu(
        TASK_MARGINS[task_name]
        - positive_similarity
        + negative_similarity
    ).mean()

    distillation_loss = encoded_batch[
        "distillation_loss"
    ]

    total_loss = (
        INFO_NCE_WEIGHT
        * info_nce_loss
        +
        RANKING_WEIGHT
        * ranking_loss
        +
        DISTILLATION_WEIGHT
        * distillation_loss
    )

    return {
        "total_loss": total_loss,
        "info_nce_loss": (
            info_nce_loss
        ),
        "ranking_loss": ranking_loss,
        "distillation_loss": (
            distillation_loss
        ),
        "positive_similarity": (
            positive_similarity.mean()
        ),
        "negative_similarity": (
            negative_similarity.mean()
        ),
        "positive_wins": (
            (
                positive_similarity
                >
                negative_similarity
            )
            .float()
            .mean()
        )
    }


# ------------------------------------------------------------
# One training epoch
# ------------------------------------------------------------

def train_one_task_epoch(
    model,
    task_name,
    data_loader,
    optimizer
):

    model.train()

    accumulated = {
        "total_loss": 0.0,
        "info_nce_loss": 0.0,
        "ranking_loss": 0.0,
        "distillation_loss": 0.0,
        "positive_similarity": 0.0,
        "negative_similarity": 0.0,
        "positive_wins": 0.0
    }

    number_of_batches = 0

    for batch in data_loader:

        optimizer.zero_grad(
            set_to_none=True
        )

        encoded_batch = (
            encode_training_batch(
                model,
                task_name,
                batch
            )
        )

        losses = calculate_task_loss(
            encoded_batch,
            task_name
        )

        losses["total_loss"].backward()

        torch.nn.utils.clip_grad_norm_(
            [
                parameter
                for parameter
                in model.parameters()
                if parameter.requires_grad
            ],
            max_norm=GRADIENT_CLIP_NORM
        )

        optimizer.step()

        for metric_name in accumulated:

            accumulated[
                metric_name
            ] += float(
                losses[
                    metric_name
                ].detach()
            )

        number_of_batches += 1

    return {
        metric_name: (
            metric_value
            / number_of_batches
        )
        for metric_name, metric_value
        in accumulated.items()
    }


# ------------------------------------------------------------
# Validation comparison tuple
#
# Priority:
# 1. Recall@1
# 2. MRR
# 3. Data accuracy
# 4. Data gap
# ------------------------------------------------------------

def validation_priority(
    metrics
):

    return (
        round(
            metrics["recall_at_1"],
            8
        ),
        round(
            metrics["mrr"],
            8
        ),
        round(
            metrics["data_accuracy"],
            8
        ),
        round(
            metrics["mean_data_gap"],
            8
        )
    )


# ------------------------------------------------------------
# Train a single task-specific head
# ------------------------------------------------------------

def train_task_head(
    model,
    task_name
):

    print(
        "\n"
        + "=" * 80
    )

    print(
        "Training:",
        task_name
    )

    print(
        "=" * 80
    )

    active_parameters = (
        activate_task_parameters(
            model,
            task_name
        )
    )

    optimizer = torch.optim.AdamW(
        active_parameters,
        lr=TASK_LEARNING_RATES[
            task_name
        ],
        weight_decay=WEIGHT_DECAY
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=SCHEDULER_PATIENCE,
            min_lr=1e-6
        )
    )

    initial_metrics = (
        evaluate_v2_validation(
            model,
            task_name
        )
    )

    best_metrics = deepcopy(
        initial_metrics
    )

    best_priority = (
        validation_priority(
            best_metrics
        )
    )

    best_task_state = (
        get_task_state(
            model,
            task_name
        )
    )

    best_epoch = 0
    epochs_without_improvement = 0

    task_history = []

    task_start_time = time.time()

    for epoch in range(
        1,
        MAXIMUM_EPOCHS + 1
    ):

        epoch_start_time = time.time()

        train_metrics = (
            train_one_task_epoch(
                model,
                task_name,
                train_loaders[
                    task_name
                ],
                optimizer
            )
        )

        validation_metrics = (
            evaluate_v2_validation(
                model,
                task_name
            )
        )

        current_priority = (
            validation_priority(
                validation_metrics
            )
        )

        current_learning_rate = (
            optimizer.param_groups[0][
                "lr"
            ]
        )

        selection_score = (
            0.5
            * validation_metrics[
                "recall_at_1"
            ]
            +
            0.25
            * validation_metrics[
                "mrr"
            ]
            +
            0.25
            * validation_metrics[
                "data_accuracy"
            ]
        )

        scheduler.step(
            selection_score
        )

        improved = (
            current_priority
            > best_priority
        )

        if improved:

            best_priority = (
                current_priority
            )

            best_metrics = deepcopy(
                validation_metrics
            )

            best_task_state = (
                get_task_state(
                    model,
                    task_name
                )
            )

            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(
                {
                    "task": task_name,
                    "epoch": epoch,
                    "model_state_dict": (
                        best_task_state
                    ),
                    "validation_metrics": (
                        best_metrics
                    ),
                    "training_metrics": (
                        train_metrics
                    )
                },
                V2_MODEL_DIR
                / f"best_{task_name}_head.pt"
            )

        else:

            epochs_without_improvement += 1

        epoch_seconds = (
            time.time()
            - epoch_start_time
        )

        history_record = {
            "task": task_name,
            "epoch": epoch,
            "learning_rate": (
                current_learning_rate
            ),
            "train_total_loss": (
                train_metrics[
                    "total_loss"
                ]
            ),
            "train_info_nce_loss": (
                train_metrics[
                    "info_nce_loss"
                ]
            ),
            "train_ranking_loss": (
                train_metrics[
                    "ranking_loss"
                ]
            ),
            "train_distillation_loss": (
                train_metrics[
                    "distillation_loss"
                ]
            ),
            "train_positive_similarity": (
                train_metrics[
                    "positive_similarity"
                ]
            ),
            "train_negative_similarity": (
                train_metrics[
                    "negative_similarity"
                ]
            ),
            "train_positive_wins": (
                train_metrics[
                    "positive_wins"
                ]
            ),
            "validation_recall_at_1": (
                validation_metrics[
                    "recall_at_1"
                ]
            ),
            "validation_recall_at_5": (
                validation_metrics[
                    "recall_at_5"
                ]
            ),
            "validation_mrr": (
                validation_metrics["mrr"]
            ),
            "validation_ndcg_at_10": (
                validation_metrics[
                    "ndcg_at_10"
                ]
            ),
            "validation_data_accuracy": (
                validation_metrics[
                    "data_accuracy"
                ]
            ),
            "validation_data_gap": (
                validation_metrics[
                    "mean_data_gap"
                ]
            ),
            "selection_score": (
                selection_score
            ),
            "improved": improved,
            "epoch_seconds": (
                epoch_seconds
            )
        }

        task_history.append(
            history_record
        )

        print(
            f"{task_name} | "
            f"epoch {epoch:02d} | "
            f"loss {train_metrics['total_loss']:.4f} | "
            f"R@1 {validation_metrics['recall_at_1']:.4f} | "
            f"MRR {validation_metrics['mrr']:.4f} | "
            f"data acc "
            f"{validation_metrics['data_accuracy']:.4f} | "
            f"gap "
            f"{validation_metrics['mean_data_gap']:.4f} | "
            f"lr {current_learning_rate:.2e}"
        )

        if improved:

            print(
                "  Saved new best task head."
            )

        else:

            print(
                "  Epochs without improvement:",
                epochs_without_improvement
            )

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):

            print(
                f"Early stopping {task_name} "
                f"after epoch {epoch}."
            )

            break

    # Restore the strongest state, including the
    # initial baseline if training never improved.
    load_task_state(
        model,
        best_task_state
    )

    task_minutes = (
        time.time()
        - task_start_time
    ) / 60.0

    print(
        f"\nBest {task_name} epoch:",
        best_epoch
    )

    print(
        "Best validation metrics:"
    )

    display(
        pd.DataFrame(
            [best_metrics]
        ).round(6)
    )

    print(
        "Training minutes:",
        round(task_minutes, 2)
    )

    return {
        "task": task_name,
        "best_epoch": best_epoch,
        "initial_metrics": (
            initial_metrics
        ),
        "best_metrics": best_metrics,
        "history": task_history,
        "training_minutes": task_minutes
    }


# ------------------------------------------------------------
# Train all three independent heads
# ------------------------------------------------------------

all_training_results = {}
all_history_records = []

overall_start_time = time.time()

for task_name in TASKS:

    task_result = train_task_head(
        v2_model,
        task_name
    )

    all_training_results[
        task_name
    ] = task_result

    all_history_records.extend(
        task_result["history"]
    )


overall_training_minutes = (
    time.time()
    - overall_start_time
) / 60.0


# Re-enable parameters after training
for parameter in v2_model.parameters():

    parameter.requires_grad = True


# ------------------------------------------------------------
# Save combined best model
# ------------------------------------------------------------

COMBINED_MODEL_PATH = (
    V2_MODEL_DIR
    / "best_task_specific_model.pt"
)

torch.save(
    {
        "model_state_dict": (
            v2_model.state_dict()
        ),
        "architecture": (
            "TaskSpecificFusionModel"
        ),
        "best_epochs": {
            task_name: int(
                result["best_epoch"]
            )
            for task_name, result
            in all_training_results.items()
        },
        "best_validation_metrics": {
            task_name: result[
                "best_metrics"
            ]
            for task_name, result
            in all_training_results.items()
        },
        "training_configuration": {
            "maximum_epochs": (
                MAXIMUM_EPOCHS
            ),
            "early_stopping_patience": (
                EARLY_STOPPING_PATIENCE
            ),
            "learning_rates": (
                TASK_LEARNING_RATES
            ),
            "temperatures": (
                TASK_TEMPERATURES
            ),
            "margins": TASK_MARGINS,
            "info_nce_weight": (
                INFO_NCE_WEIGHT
            ),
            "ranking_weight": (
                RANKING_WEIGHT
            ),
            "distillation_weight": (
                DISTILLATION_WEIGHT
            ),
            "random_seed": RANDOM_SEED
        }
    },
    COMBINED_MODEL_PATH
)


# ------------------------------------------------------------
# Save histories and summaries
# ------------------------------------------------------------

training_history_df = pd.DataFrame(
    all_history_records
)

training_history_df.to_csv(
    V2_REPORT_DIR
    / "training_history.csv",
    index=False
)

training_history_df.to_parquet(
    V2_REPORT_DIR
    / "training_history.parquet",
    index=False
)


training_summary = {
    "overall_training_minutes": (
        overall_training_minutes
    ),
    "tasks": {
        task_name: {
            "best_epoch": int(
                result["best_epoch"]
            ),
            "initial_metrics": (
                result[
                    "initial_metrics"
                ]
            ),
            "best_metrics": (
                result[
                    "best_metrics"
                ]
            ),
            "training_minutes": (
                result[
                    "training_minutes"
                ]
            )
        }
        for task_name, result
        in all_training_results.items()
    }
}

with open(
    V2_REPORT_DIR
    / "training_summary.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        training_summary,
        file,
        indent=2
    )


# ------------------------------------------------------------
# Final validation comparison
# ------------------------------------------------------------

validation_comparison_records = []

for task_name in TASKS:

    initial_metrics = (
        all_training_results[
            task_name
        ]["initial_metrics"]
    )

    best_metrics = (
        all_training_results[
            task_name
        ]["best_metrics"]
    )

    validation_comparison_records.append(
        {
            "task": task_name,
            "initial_recall_at_1": (
                initial_metrics[
                    "recall_at_1"
                ]
            ),
            "best_recall_at_1": (
                best_metrics[
                    "recall_at_1"
                ]
            ),
            "recall_at_1_change": (
                best_metrics[
                    "recall_at_1"
                ]
                -
                initial_metrics[
                    "recall_at_1"
                ]
            ),
            "initial_mrr": (
                initial_metrics["mrr"]
            ),
            "best_mrr": (
                best_metrics["mrr"]
            ),
            "mrr_change": (
                best_metrics["mrr"]
                -
                initial_metrics["mrr"]
            ),
            "initial_data_accuracy": (
                initial_metrics[
                    "data_accuracy"
                ]
            ),
            "best_data_accuracy": (
                best_metrics[
                    "data_accuracy"
                ]
            ),
            "data_accuracy_change": (
                best_metrics[
                    "data_accuracy"
                ]
                -
                initial_metrics[
                    "data_accuracy"
                ]
            ),
            "best_epoch": (
                all_training_results[
                    task_name
                ]["best_epoch"]
            )
        }
    )


validation_comparison_df = pd.DataFrame(
    validation_comparison_records
)

print(
    "\nFinal Version 2 validation comparison"
)

display(
    validation_comparison_df.round(6)
)


validation_comparison_df.to_csv(
    V2_REPORT_DIR
    / "validation_comparison.csv",
    index=False
)


print("\nVersion 2 training completed")

print(
    "Combined model:",
    COMBINED_MODEL_PATH
)

print(
    "Training history:",
    V2_REPORT_DIR
    / "training_history.csv"
)

print(
    "Training summary:",
    V2_REPORT_DIR
    / "training_summary.json"
)

print(
    "Total training minutes:",
    round(
        overall_training_minutes,
        2
    )
)



Training: query_chart


Encoding query→chart queries:   0%|          | 0/2 [00:00<?, ?it/s]

Encoding query→chart charts:   0%|          | 0/9 [00:00<?, ?it/s]

Encoding query→chart queries:   0%|          | 0/2 [00:00<?, ?it/s]

Encoding query→chart charts:   0%|          | 0/9 [00:00<?, ?it/s]

query_chart | epoch 01 | loss 4.4912 | R@1 0.4371 | MRR 0.5775 | data acc 0.6159 | gap 0.0015 | lr 2.00e-04
  Epochs without improvement: 1


... (135 outputs left out)

,number_of_queries,recall_at_1,recall_at_5,recall_at_10,mrr,ndcg_at_10,median_positive_rank,mean_data_gap,data_accuracy
0,346,0.956647,0.982659,0.982659,0.969123,0.971975,1.0,0.383117,0.962428


Training minutes: 0.04

Final Version 2 validation comparison


,task,initial_recall_at_1,best_recall_at_1,recall_at_1_change,initial_mrr,best_mrr,mrr_change,initial_data_accuracy,best_data_accuracy,data_accuracy_change,best_epoch
0,query_chart,0.443709,0.443709,0.0,0.579898,0.579898,0.000000,0.615894,0.615894,0.0,0
1,chart_chart,0.927746,0.927746,0.0,0.951962,0.952552,0.000590,0.930636,0.930636,0.0,38
2,chart_table,0.956647,0.956647,0.0,0.969122,0.969123,0.000001,0.962428,0.962428,0.0,1



Version 2 training completed
Combined model: /home/pnola130/projects/chart_data_retrieval/artifacts/models/task_specific_fusion_v2/best_task_specific_model.pt
Training history: /home/pnola130/projects/chart_data_retrieval/reports/experiments/v2_task_specific_heads/training_history.csv
Training summary: /home/pnola130/projects/chart_data_retrieval/reports/experiments/v2_task_specific_heads/training_summary.json
Total training minutes: 0.33


In [4]:
# ============================================================
# BLOCK 5 — Finalize and archive the V2 architecture experiment
# ============================================================

from pathlib import Path
import hashlib
import json
import shutil

import pandas as pd


PROJECT_ROOT = (
    Path.home()
    / "projects"
    / "chart_data_retrieval"
)

V2_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "experiments"
    / "v2_task_specific_heads"
)

V2_MODEL_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "models"
    / "task_specific_fusion_v2"
)

V2_ARCHIVE_MODEL_DIR = (
    V2_REPORT_DIR
    / "model_checkpoints"
)

V2_ARCHIVE_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Load experiment results
# ------------------------------------------------------------

initial_metrics_path = (
    V2_REPORT_DIR
    / "initial_validation_metrics.json"
)

training_summary_path = (
    V2_REPORT_DIR
    / "training_summary.json"
)

training_history_path = (
    V2_REPORT_DIR
    / "training_history.csv"
)

assert initial_metrics_path.exists()
assert training_summary_path.exists()
assert training_history_path.exists()

with open(initial_metrics_path, "r") as file:
    initial_metrics = json.load(file)

with open(training_summary_path, "r") as file:
    training_summary = json.load(file)

training_history_df = pd.read_csv(
    training_history_path
)


# ------------------------------------------------------------
# Copy model files into the experiment archive
# ------------------------------------------------------------

model_files_to_archive = [
    "initial_model.pt",
    "best_task_specific_model.pt",
    "best_query_chart_head.pt",
    "best_chart_chart_head.pt",
    "best_chart_table_head.pt",
    "architecture_config.json",
]

archived_model_files = []

for file_name in model_files_to_archive:

    source_path = (
        V2_MODEL_DIR
        / file_name
    )

    if not source_path.exists():
        continue

    destination_path = (
        V2_ARCHIVE_MODEL_DIR
        / file_name
    )

    shutil.copy2(
        source_path,
        destination_path
    )

    archived_model_files.append(
        destination_path
    )


# ------------------------------------------------------------
# Experiment conclusions
# ------------------------------------------------------------

conclusions = {
    "experiment_name": "Version 2 — Task-Specific Fusion Heads",
    "benchmark_sources": 700,
    "training_objective": (
        "Replace the single shared fusion head with separate "
        "query-chart, chart-chart, and chart-table heads."
    ),
    "query_chart_conclusion": (
        "No validation improvement. The baseline-preserving "
        "epoch-0 model remained the best checkpoint."
    ),
    "chart_chart_conclusion": (
        "Recall@1 and data accuracy were unchanged. "
        "MRR improved by only 0.000590, which is not "
        "a meaningful practical improvement."
    ),
    "chart_table_conclusion": (
        "Recall@1 and data accuracy were unchanged. "
        "The MRR change was approximately 0.000001."
    ),
    "overall_conclusion": (
        "Task-specific heads prevented the cross-task degradation "
        "observed with the Version 1 shared fusion head, but training "
        "did not materially improve retrieval over the initialized "
        "single-task baselines."
    ),
    "next_experiment": (
        "Version 2B — stronger and more realistic hard-negative "
        "generation, followed by retraining on the same controlled "
        "benchmark before scaling the dataset."
    ),
    "test_set_evaluated": False,
    "reason_test_not_evaluated": (
        "No meaningful validation improvement was observed, so "
        "additional test-set evaluation would not guide model selection."
    ),
}


# ------------------------------------------------------------
# Write the Markdown report
# ------------------------------------------------------------

report_text = """# Version 2 Experiment Report

## Experiment

Version 2 replaced the single shared fusion representation with three
task-specific retrieval heads:

1. Query → Chart
2. Chart → Chart
3. Chart → Table

The model used baseline-preserving initialization so that training began
from the strongest existing representation for each task.

## Validation Results

| Task | Initial R@1 | Best R@1 | Change | Initial MRR | Best MRR | Change |
|---|---:|---:|---:|---:|---:|---:|
| Query → Chart | 0.443709 | 0.443709 | 0.000000 | 0.579898 | 0.579898 | 0.000000 |
| Chart → Chart | 0.927746 | 0.927746 | 0.000000 | 0.951962 | 0.952552 | 0.000590 |
| Chart → Table | 0.956647 | 0.956647 | 0.000000 | 0.969122 | 0.969123 | 0.000001 |

## Interpretation

The task-specific architecture successfully avoided the major performance
regression produced by Version 1's shared gated-fusion head. However, the
new heads did not materially improve retrieval.

For Query → Chart, every trained checkpoint performed worse than the
initial DePlot+E5 representation, so epoch 0 was retained.

For Chart → Chart and Chart → Table, Recall@1 and counterfactual accuracy
did not improve. The very small MRR changes are too small to be considered
meaningful evidence of improvement.

## Decision

The test set was not evaluated for this experiment because model selection
must be based on validation performance, and no meaningful validation
improvement occurred.

The next experiment will improve the training signal by constructing more
realistic hard negatives:

- same title and categories but permuted numerical values;
- same schema with perturbed values;
- same values attached to different categories;
- reversed temporal trends;
- nearby numerical values that remain incorrect;
- same topic, entity, year, or answer type from another source.

This experiment is preserved as an important negative result: separating
the heads fixed task interference, but architecture changes alone were not
sufficient.
"""

report_path = (
    V2_REPORT_DIR
    / "VERSION_2_EXPERIMENT_REPORT.md"
)

report_path.write_text(
    report_text,
    encoding="utf-8"
)


# ------------------------------------------------------------
# Save conclusions and manifest
# ------------------------------------------------------------

conclusions_path = (
    V2_REPORT_DIR
    / "version_2_conclusions.json"
)

with open(conclusions_path, "w") as file:
    json.dump(
        conclusions,
        file,
        indent=2
    )


def sha256_file(file_path):

    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


manifest_records = []

for file_path in sorted(
    path
    for path in V2_REPORT_DIR.rglob("*")
    if path.is_file()
):

    if file_path.name == "manifest.json":
        continue

    manifest_records.append(
        {
            "relative_path": str(
                file_path.relative_to(
                    V2_REPORT_DIR
                )
            ),
            "size_bytes": file_path.stat().st_size,
            "sha256": sha256_file(file_path),
        }
    )

manifest = {
    "experiment": "v2_task_specific_heads",
    "status": "completed_validation_only",
    "number_of_files": len(manifest_records),
    "files": manifest_records,
}

manifest_path = (
    V2_REPORT_DIR
    / "manifest.json"
)

with open(manifest_path, "w") as file:
    json.dump(
        manifest,
        file,
        indent=2
    )


print("Version 2 archived successfully")
print("Report:", report_path)
print("Conclusions:", conclusions_path)
print("Manifest:", manifest_path)
print("Archived model files:", len(archived_model_files))
print("Total archived files:", len(manifest_records))

Version 2 archived successfully
Report: /home/pnola130/projects/chart_data_retrieval/reports/experiments/v2_task_specific_heads/VERSION_2_EXPERIMENT_REPORT.md
Conclusions: /home/pnola130/projects/chart_data_retrieval/reports/experiments/v2_task_specific_heads/version_2_conclusions.json
Manifest: /home/pnola130/projects/chart_data_retrieval/reports/experiments/v2_task_specific_heads/manifest.json
Archived model files: 5
Total archived files: 12
